In [ ]:
import numpy as np
from itertools import product

def log2(x):
    return np.log2(x)

def nb_filter_exp(alpha):
    return -np.log2(np.sin(alpha))

def is_valid_config(C, tol=1e-12):
    if C.shape[0] != C.shape[1]:
        return False
    if not np.allclose(C, C.T, atol=tol):
        return False
    if not np.allclose(np.diag(C), 1.0, atol=tol):
        return False
    eigvals = np.linalg.eigvalsh(C)
    return np.min(eigvals) > tol

def c_prime_matrix(C, alpha, k=4):
    s2 = np.sin(alpha) ** 2
    c2 = np.cos(alpha) ** 2
    Cp = np.zeros_like(C)
    for i in range(k):
        for j in range(k):
            if i == j:
                Cp[i, j] = 1.0
            else:
                Cp[i, j] = (C[i, j] + c2 / (k - 1)) / s2
    return Cp

def nb_rep_exp_matrix(alpha, C, Cp, k=4):
    detC = np.linalg.det(C)
    detCp = np.linalg.det(Cp)
    if detC <= 0 or detCp <= 0:
        return None
    val = (
        0.5 * np.log2(detC)
        - (k - 1) * np.log2(np.sin(alpha))
        - 0.5 * np.log2(detCp)
    )
    return max(0.0, val)

def qrw_cost_N_units(n1, Cprime, c, c1, c2, rho):
    if abs(Cprime) >= 1:
        return None, None
    log2_psol = 0.5 * np.log2(1 - Cprime ** 2)
    theta_prime = np.arccos(Cprime)

    log2_sin_beta = (c2 - c1) * n1
    if log2_sin_beta > 0:
        return None, None
    sin_beta = 2.0 ** log2_sin_beta
    cos2_beta = 1.0 - sin_beta ** 2

    denom = 1.0 + np.cos(theta_prime)
    if denom <= 0:
        return None, None
    ratio = 2.0 * cos2_beta / denom
    if ratio >= 1:
        return None, None
    log2_W = 0.5 * np.log2(1 - ratio)

    rho0 = (log2_sin_beta - log2_W) / n1
    if rho0 < 0 or rho > rho0 + 1e-9:
        return None, None

    zeta = 2 * c + log2_psol / n1
    if zeta <= 0:
        return None, None

    if zeta + rho - rho0 <= 0:
        return None, None

    log2_eps = 2 * c1 + log2_psol / n1 + rho - rho0
    S_log2 = c1 + rho
    U_log2 = max(rho, (rho + c2) / 2.0)
    delta_log2 = -c1

    inv_sqrt_eps   = -0.5 * log2_eps
    inv_sqrt_delta = -0.5 * delta_log2
    term = inv_sqrt_delta + U_log2
    T1_log2 = max(S_log2, inv_sqrt_eps + max(term, 0.0))
    FAS1_log2 = max(zeta, 0.0) + T1_log2
    NbRep_QRW_log2 = max(0.0, c - zeta)
    inner = max(1.0, (1 - c) + FAS1_log2)

    T_QRW_log2 = NbRep_QRW_log2 + inner
    diag = dict(
        c=c, c1=c1, c2=c2, rho=rho,
        rho0=rho0, zeta=zeta, log2_eps=log2_eps,
        S=S_log2, U=U_log2, delta=delta_log2,
        T1=T1_log2, FAS1=FAS1_log2,
        NbRep_QRW=NbRep_QRW_log2,
    )
    return T_QRW_log2, diag

def optimize_qrw(n1, Cprime):
    best_T, best_diag = None, None
    for c_qrw in np.linspace(0.25, 0.95, 20):
        for c1 in np.linspace(0.1, c_qrw, 20):
            for c2 in (0, c1):
                if c1 == c2:
                    continue
                for rho in (0.0, 0.05, 0.1, 0.15):
                    T, diag = qrw_cost_N_units(n1, Cprime, c_qrw, c1, c2, rho)
                    if T is None:
                        continue
                    if best_T is None or T < best_T:
                        best_T, best_diag = T, diag
    return best_T, best_diag

def full_hybrid_time(alpha0, C):
    k = 4
    s = np.sin(alpha0) ** 2

    if not is_valid_config(C):
        return None
    Cp = c_prime_matrix(C, alpha0, k)
    if not is_valid_config(Cp):
        return None

    Cprime12 = Cp[0, 1]
    Cprime34 = Cp[2, 3]
    if abs(Cprime12) >= 1 or abs(Cprime34) >= 1:
        return None

    detC = np.linalg.det(C)
    if detC <= 0:
        return None
    log2_L = -np.log2(detC) / 6.0

    log2_Ri = log2_L + 0.5 * np.log2(s)
    if log2_Ri < 0:
        return None
    n1 = log2_Ri

    log2_p12 = 0.5 * np.log2(1 - Cprime12 ** 2)
    log2_p34 = 0.5 * np.log2(1 - Cprime34 ** 2)
    E12 = 2 * log2_Ri + log2_p12
    E34 = 2 * log2_Ri + log2_p34
    if E12 < 0 or E34 < 0:
        return None

    R12_sq = 2.0 + 2.0 * Cprime12
    R34_sq = 2.0 + 2.0 * Cprime34
    if R12_sq <= 0.0 or R34_sq <= 0.0:
        return None
    r0_sq = 1.0 / s
    Ceff = (r0_sq - R12_sq - R34_sq) / (2.0 * np.sqrt(R12_sq) * np.sqrt(R34_sq))
    if abs(Ceff) >= 1:
        return None

    log2_psol3 = 0.5 * np.log2(1 - Ceff ** 2)
    m = E12 + E34 + log2_psol3
    T1234_log2 = (E12 + E34) + 0.5 * log2_psol3 if m >= 0 else max(E12, E34)

    best_T12_N, diag12 = optimize_qrw(n1, Cprime12)
    best_T34_N, diag34 = optimize_qrw(n1, Cprime34)
    if best_T12_N is None or best_T34_N is None:
        return None

    T12_log2 = max(n1 * best_T12_N, E12)
    T34_log2 = max(n1 * best_T34_N, E34)
    T_filter_log2 = max(T12_log2, T34_log2, T1234_log2)

    nbrep = nb_rep_exp_matrix(alpha0, C, Cp, k)
    if nbrep is None:
        return None

    T_hybrid_log2 = nb_filter_exp(alpha0) + nbrep + T_filter_log2

    return dict(
        Cp=Cp, Cprime12=Cprime12, Cprime34=Cprime34,
        E12=E12, E34=E34, Ceff=Ceff,
        log2_Ri=log2_Ri, n1=n1,
        log2_p12=log2_p12, log2_p34=log2_p34,
        log2_psol3=log2_psol3, m=m,
        nb_filter=nb_filter_exp(alpha0), nb_rep=nbrep,
        T12=T12_log2, T34=T34_log2, T1234=T1234_log2,
        T_filter=T_filter_log2, T_hybrid=T_hybrid_log2,
        qrw12=diag12, qrw34=diag34,
    )

# ----------------------------------------------------------------
# Trade-off sweep over (C12, C13, C23) with CL23 constraint
# ----------------------------------------------------------------
if __name__ == "__main__":
    # ---- memory bins ----
    NUM_MEM = 10
    mem_bins = np.linspace(0.1724, 0.2100, NUM_MEM)
    T_hyb = np.full(NUM_MEM, np.inf)
    info  = [None] * NUM_MEM

    def assign_bin(edges, val):
        i = np.searchsorted(edges, val, side='right') - 1
        return max(0, min(i, len(edges) - 2))

    # ---- alpha grid: excluded pi/2, focused on useful region ----
    alpha_grid = np.linspace(1.20, 1.48, 10)

    # ---- configuration grids: denser near balanced -0.25 ----
    C12_grid = np.concatenate([
        np.linspace(-0.5, -0.36, 5),
        np.linspace(-0.36, -0.20, 10),
    ])
    C13_grid = np.linspace(-0.32, -0.12, 12)
    C23_grid = np.linspace(-0.32, -0.12, 12)

    count_feasible = 0
    count_binned   = 0

    for alpha0 in alpha_grid:
        for C12 in C12_grid:
            for C13 in C13_grid:
                for C23 in C23_grid:
                    C34 = -1.5 - C12 - 2 * C13 - 2 * C23
                    if C34 <= -1.0 or C34 >= 0.0:
                        continue

                    C = np.array([
                        [1.0, C12, C13, C13],
                        [C12, 1.0, C23, C23],
                        [C13, C23, 1.0, C34],
                        [C13, C23, C34, 1.0],
                    ])

                    res = full_hybrid_time(alpha0, C)
                    if res is None or res['m'] < 0:
                        continue
                    count_feasible += 1

                    detC = np.linalg.det(C)
                    if detC <= 0:
                        continue
                    memory_outer = -np.log2(detC) / 6.0
                    memory_peak  = max(memory_outer, res['E12'], res['E34'])

                    if memory_peak < mem_bins[0] or memory_peak > mem_bins[-1]:
                        continue
                    i = assign_bin(mem_bins, memory_peak)
                    count_binned += 1

                    if res['T_hybrid'] < T_hyb[i]:
                        T_hyb[i] = res['T_hybrid']
                        info[i]  = (alpha0, C12, C13, C23, C34,
                                    memory_outer, memory_peak, res)

    # ---- balanced configuration as an extra anchor ----
    C_bal = np.array([
        [1.0, -0.25, -0.25, -0.25],
        [-0.25, 1.0, -0.25, -0.25],
        [-0.25, -0.25, 1.0, -0.25],
        [-0.25, -0.25, -0.25, 1.0],
    ])
    mem_bal_outer = -np.log2(np.linalg.det(C_bal)) / 6.0
    best_bal = None
    for alpha0 in alpha_grid:
        res = full_hybrid_time(alpha0, C_bal)
        if res is None or res['m'] < 0:
            continue
        if best_bal is None or res['T_hybrid'] < best_bal[0]:
            best_bal = (res['T_hybrid'], alpha0, res)
    if best_bal is not None:
        mem_bal_peak = max(mem_bal_outer,
                           best_bal[2]['E12'],
                           best_bal[2]['E34'])
        if mem_bins[0] <= mem_bal_peak <= mem_bins[-1]:
            i = assign_bin(mem_bins, mem_bal_peak)
            if best_bal[0] < T_hyb[i]:
                T_hyb[i] = best_bal[0]
                info[i]  = (best_bal[1], -0.25, -0.25, -0.25, -0.25,
                            mem_bal_outer, mem_bal_peak, best_bal[2])
        else:
            print(f"NOTE: balanced anchor peak memory = {mem_bal_peak:.4f} "
                  f"is outside bin range [{mem_bins[0]:.4f}, {mem_bins[-1]:.4f}]")
    else:
        print("WARNING: balanced anchor infeasible on the alpha grid")

    # ================================================================
    # Print the table
    # ================================================================
    print("=" * 130)
    print("HYBRID 4-SIEVE TRADE-OFF (peak-memory binning)")
    print("=" * 130)
    print(f"feasible configs (M>=1): {count_feasible}")
    print(f"binned into memory bins: {count_binned}")
    print()

    header = (
        f"{'mem outer':>9} {'mem peak':>9} {'T_hybrid':>10} {'alpha0':>8} "
        f"{'C12':>7} {'C13':>7} {'C23':>7} {'C34':>7} "
        f"{'|R_i|':>7} {'E12':>7} {'E34':>7} {'m':>7} "
        f"{'Ceff':>7} {'nbfilt':>7} {'nbrep':>7} | "
        f"{'c12':>6} {'c1_12':>6} {'c2_12':>6} {'rho12':>6} "
        f"{'rho0_12':>8} {'zeta12':>8} | "
        f"{'c34':>6} {'c1_34':>6} {'c2_34':>6} {'rho34':>6} "
        f"{'rho0_34':>8} {'zeta34':>8}"
    )
    print(header)
    print("-" * 130)

    for inf in info:
        if inf is None:
            continue
        a0_, c12_, c13_, c23_, c34_, mouter_, mpeak_, r_ = inf
        q12 = r_['qrw12']
        q34 = r_['qrw34']

        print(
            f"{mouter_:>9.4f} {mpeak_:>9.4f} {r_['T_hybrid']:>10.4f} {a0_:>8.4f} "
            f"{c12_:>7.4f} {c13_:>7.4f} {c23_:>7.4f} {c34_:>7.4f} "
            f"{r_['log2_Ri']:>7.4f} {r_['E12']:>7.4f} {r_['E34']:>7.4f} "
            f"{r_['m']:>7.4f} {r_['Ceff']:>7.4f} "
            f"{r_['nb_filter']:>7.4f} {r_['nb_rep']:>7.4f} | "
            f"{q12['c']:>6.3f} {q12['c1']:>6.3f} {q12['c2']:>6.3f} "
            f"{q12['rho']:>6.3f} {q12['rho0']:>8.4f} {q12['zeta']:>8.4f} | "
            f"{q34['c']:>6.3f} {q34['c1']:>6.3f} {q34['c2']:>6.3f} "
            f"{q34['rho']:>6.3f} {q34['rho0']:>8.4f} {q34['zeta']:>8.4f}"
        )

In [10]:
print(header)
print("-" * 130)

# collect non-empty rows and sort
rows = [inf for inf in info if inf is not None]
rows.sort(key=lambda x: x[5])   # sort by outer memory
# rows.sort(key=lambda x: x[6])   # uncomment to sort by peak memory
# rows.sort(key=lambda x: x[7]['T_hybrid'])   # uncomment to sort by time

for inf in rows:
    a0_, c12_, c13_, c23_, c34_, mouter_, mpeak_, r_ = inf
    q12 = r_['qrw12']
    q34 = r_['qrw34']

    print(
        f"{mouter_:>9.4f} {mpeak_:>9.4f} {r_['T_hybrid']:>10.4f} {a0_:>8.4f} "
        f"{c12_:>7.4f} {c13_:>7.4f} {c23_:>7.4f} {c34_:>7.4f} "
        f"{r_['log2_Ri']:>7.4f} {r_['E12']:>7.4f} {r_['E34']:>7.4f} "
        f"{r_['m']:>7.4f} {r_['Ceff']:>7.4f} "
        f"{r_['nb_filter']:>7.4f} {r_['nb_rep']:>7.4f} | "
        f"{q12['c']:>6.3f} {q12['c1']:>6.3f} {q12['c2']:>6.3f} "
        f"{q12['rho']:>6.3f} {q12['rho0']:>8.4f} {q12['zeta']:>8.4f} | "
        f"{q34['c']:>6.3f} {q34['c1']:>6.3f} {q34['c2']:>6.3f} "
        f"{q34['rho']:>6.3f} {q34['rho0']:>8.4f} {q34['zeta']:>8.4f}"
    )

mem outer  mem peak   T_hybrid   alpha0     C12     C13     C23     C34   |R_i|     E12     E34       m    Ceff  nbfilt   nbrep |    c12  c1_12  c2_12  rho12  rho0_12   zeta12 |    c34  c1_34  c2_34  rho34  rho0_34   zeta34
----------------------------------------------------------------------------------------------------------------------------------
   0.1748    0.1748     0.3393   1.3067 -0.3240 -0.2143 -0.2143 -0.3189  0.1239  0.1682  0.1710  0.0103 -0.6051  0.0509  0.1137 |  0.434  0.100  0.000  0.000   0.2007   0.2257 |  0.434  0.100  0.000  0.000   0.1982   0.2486
   0.1754    0.1754     0.3415   1.3067 -0.3240 -0.2314 -0.1971 -0.3189  0.1244  0.1692  0.1721  0.0123 -0.6051  0.0509  0.1138 |  0.434  0.100  0.000  0.000   0.2007   0.2284 |  0.471  0.100  0.000  0.000   0.1982   0.3249
   0.1764    0.1764     0.3448   1.3067 -0.3480 -0.2143 -0.1971 -0.3291  0.1255  0.1572  0.1684  0.0115 -0.5942  0.0509  0.1144 |  0.508  0.100  0.000  0.000   0.2130   0.2689 |  0.434  0.100  0.00

In [29]:
# import numpy as np
# import matplotlib.pyplot as plt

# hybrid = {
#     "memory": np.array([0.1766, 0.1772, 0.1810, 0.1858]),
#     "T":      np.array([0.3087, 0.3113, 0.3065, 0.3082]),
# }


# # Classical 4-sieve (CL23, Table 3)
# mem_class = np.array([0.1724, 0.1750, 0.1800, 0.1887,
#                       0.1930, 0.1980, 0.2030, 0.2075])
# T_class   = np.array([0.4180, 0.3800, 0.3520, 0.3240,
#                       0.3150, 0.3060, 0.2980, 0.2925])

# # Pure quantum 4-sieve (CL23, Proposition 17 & 18)
# mem_q = np.array([0.1724, 0.175, 0.18, 0.1813])
# T_q   = np.array([0.3276, 0.3153, 0.3127, 0.3120])

# # ================================================================
# # Plot
# # ================================================================
# fig, ax = plt.subplots(figsize=(8.5, 6))

# # classical curve
# ax.plot(mem_class, T_class, 'o-', color='#1f77b4',
#         label='Classical 4-sieve (CL23)',
#         markersize=4, linewidth=1.8)

# # pure quantum curve (only two points → dashed)
# ax.plot(mem_q, T_q, 's--', color='#2ca02c',
#         label='Pure quantum 4-sieve (CL23)',
#         markersize=4, linewidth=2)

# # hybrid curve
# ax.plot(hybrid["memory"], hybrid["T"], 'D-', color='#d62728',
#         label='Hybrid quantum 4-sieve (ours)',
#         markersize=4, linewidth=2.2)

# i_best = 2  # the m=0.1810 point
# ax.plot(hybrid["memory"][i_best], hybrid["T"][i_best],
#         'D', color='#d62728', markersize=12,
#         markerfacecolor='none', markeredgewidth=2)
# ax.annotate(f"best: ({hybrid['memory'][i_best]:.4f}, {hybrid['T'][i_best]:.4f})",
#             xy=(hybrid["memory"][i_best], hybrid["T"][i_best]),
#             xytext=(hybrid["memory"][i_best] + 0.002,
#                     hybrid["T"][i_best] - 0.008),
#             fontsize=10, color='#d62728',
#             arrowprops=dict(arrowstyle='->', color='#d62728'))

# # mark the optimum
# i_best = int(np.argmin(hybrid["T"]))

# # axes
# ax.set_xlabel(r'Classical memory exponent $m$  ($M = 2^{m d}$)',
#               fontsize=12)
# ax.set_ylabel(r'Time exponent $t$  ($T = 2^{t d}$)', fontsize=12)
# ax.set_title('Time–memory trade-off for the 4-sieve', fontsize=13)
# ax.grid(True, alpha=0.3)
# ax.legend(loc='upper right', fontsize=11)

# ax.set_xlim(0.170, 0.215)
# ax.set_ylim(0.28, 0.44)

# plt.tight_layout()
# plt.savefig('tradeoff_4sieve.pdf', dpi=150, bbox_inches='tight')
# plt.savefig('tradeoff_4sieve.png', dpi=150, bbox_inches='tight')
# plt.show()